# ANT — A100 GPU Training

**828K parameter looping transformer with persistent external memory.**

Trained to speak **markdown** and **shell** — the two core output formats.

Optimized for A100 40GB:
- **bf16 mixed precision** (A100 native support, 2x throughput)
- **Large batches** (B=256+, saturate GPU compute)
- **Wikipedia markdown** + shell commands as LM training data
- **torch.compile** with CUDA backend (inductor fusion)
- **Checkpoints to Hugging Face Hub** (`kaaninel/ANT`)
- **Verbose logging** with TensorBoard

## Training Curriculum
1. **Phase A** (warmup): LM on QA patterns, passage in context
2. **Phase D1** (frozen encoder): Memory QA + LM, encoder frozen
3. **Phase D2** (co-adaptation): Full differentiable training
4. **Phase LM** (fluency): Extended pure LM on markdown + shell data

## Configuration
All parameters are in the **CONFIG** cell (Section 2). Edit and re-run that cell\n",
    "to change batch sizes, learning rates, steps, etc. on the fly.

## 1. Environment Setup

In [ ]:
# Verify GPU
!nvidia-smi
import torch
print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
    print(f"BF16 support: {torch.cuda.is_bf16_supported()}")

In [ ]:
# Install dependencies & login to Hugging Face
!pip install -q datasets numpy torch tensorboard huggingface_hub safetensors

from huggingface_hub import login, HfApi, create_repo
from google.colab import userdata

# Login to HF Hub — set HF_TOKEN in Colab secrets (key icon in sidebar)
try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
    print('✓ Logged in to Hugging Face Hub')
except Exception:
    print('⚠ HF_TOKEN not found in Colab secrets. Run: login() manually')
    login()

HF_REPO = 'kaaninel/ANT'
try:
    create_repo(HF_REPO, exist_ok=True, repo_type='model')
    print(f'✓ HF repo ready: https://huggingface.co/{HF_REPO}')
except Exception as e:
    print(f'HF repo note: {e}')

hf_api = HfApi()

# Local dirs for checkpoints and logs
LOCAL_DIR = '/content/ant_checkpoints'
LOG_DIR = '/content/ant_logs'
!mkdir -p {LOCAL_DIR} {LOG_DIR}
print(f'Local checkpoint dir: {LOCAL_DIR}')

In [ ]:
# Clone repository
import os
REPO_DIR = '/content/ANT'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/kaaninel/ANT.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

os.chdir(REPO_DIR)

## 2. Configuration

In [ ]:
# ═══════════════════════════════════════════════════════════
# Training Configuration — EDIT AND RE-RUN THIS CELL
# to change parameters on the fly between phases.
# ═══════════════════════════════════════════════════════════

CONFIG = {
    # ─── Data ───────────────────────────────────────────────
    'n_train': 50000,          # QA training examples
    'n_val': 2000,             # QA validation examples
    'n_shell': 20000,          # Shell command examples for LM
    'n_wiki_md': 100000,       # Wikipedia markdown sections for LM
    'wiki_min_len': 50,        # Min chars per wiki section
    'wiki_max_len': 800,       # Max chars per wiki section
    
    # ─── Architecture (828K params) ─────────────────────────
    'window_size': 8,          # Sliding window size
    'num_passes': 4,           # Diffusion passes
    
    # ─── Training schedule ──────────────────────────────────
    # Model is tiny (828K) — A100 needs HUGE batches to saturate.
    # We use gradient accumulation to reach effective batch sizes
    # that would otherwise not fit in a single allocation.
    'phase_a_steps': 1000,     # LM warmup
    'multitask_steps': 10000,  # QA + LM multi-task
    'lm_only_steps': 20000,    # Extended pure LM for fluency
    
    # ─── Batch sizes (A100-saturating) ──────────────────────
    # Phase A & Multitask LM: pure forward pass, no sliding window
    'batch_size': 2048,        # LM micro-batch per accumulation step
    'grad_accum_steps': 4,     # Effective batch = batch_size × grad_accum
    # QA: sliding window is expensive, use smaller batch
    'qa_batch_size': 512,      # QA micro-batch (sliding window encode)
    # Pure LM phase: no QA overhead, go even bigger
    'lm_batch_size': 4096,     # Pure LM micro-batch
    'lm_grad_accum_steps': 2,  # Effective = 8192
    
    'lr': 3e-4,                # Higher LR for larger effective batch
    'lm_lr': 2e-4,             # Pure LM learning rate
    'lm_weight': 0.7,          # LM loss weight (QA converges fast)
    'qa_weight': 0.3,          # QA loss weight
    
    # ─── A100 optimizations ─────────────────────────────────
    'use_bf16': True,          # BF16 mixed precision (A100 Tensor Cores)
    'use_compile': True,       # torch.compile with inductor
    'compile_mode': 'max-autotune',  # Full kernel fusion + autotuning
    'num_workers': 4,          # DataLoader workers
    'pin_memory': True,
    'cudnn_benchmark': True,   # Autotune cuDNN kernels
    
    # ─── Checkpointing & logging ───────────────────────────
    'checkpoint_interval': 500,    # Save every N steps
    'eval_interval': 250,          # Evaluate every N steps
    'log_interval': 25,            # Print every N steps
    'keep_last_n_checkpoints': 10, # Keep last N on disk
    'hf_upload_interval': 1000,    # Push to HF Hub every N steps
    
    # ─── Misc ──────────────────────────────────────────────
    'seed': 42,
    'tagged': True,            # Source provenance tags
    'max_seq_len': 384,        # Longer sequences for markdown
}

# A100 tuning
if CONFIG['cudnn_benchmark']:
    torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision('high')

eff_lm_batch = CONFIG['batch_size'] * CONFIG['grad_accum_steps']
eff_lm_only = CONFIG['lm_batch_size'] * CONFIG['lm_grad_accum_steps']
tok_per_step = eff_lm_batch * CONFIG['max_seq_len']

print('═' * 60)
print('  Configuration (edit above and re-run to change)')
print('═' * 60)
for k, v in CONFIG.items():
    print(f"  {k:30s} = {v}")
print('─' * 60)
print(f"  {'Effective LM batch':30s} = {eff_lm_batch}")
print(f"  {'Effective LM-only batch':30s} = {eff_lm_only}")
print(f"  {'Tokens per optimizer step':30s} = {tok_per_step:,}")
print('═' * 60)

## 3. Model & Data Setup

In [ ]:
import math
import time
import json
import random
from collections import Counter

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import GradScaler, autocast

# Set seeds
torch.manual_seed(CONFIG['seed'])
random.seed(CONFIG['seed'])
np.random.seed(CONFIG['seed'])

device = 'cuda'

# Import project modules
from config import ModelConfig, MemoryConfig
from model import ANT
from train_micro import (
    VOCAB, VOCAB_SIZE, PAD_ID, BOS_ID, EOS_ID, ANS_ID, ID2WORD,
    tokenize, detokenize,
    generate_dataset, generate_shell_texts, load_wikipedia_sentences,
    tag_text, tag_passage,
    TextLMDataset, ContextQADataset, MemoryQADataset,
    encode_sentence_frozen, encode_sentence_differentiable,
    sliding_lm_encode, evaluate_sliding_lm, evaluate_context_qa,
    log_model_summary, log_gradient_stats, TrainingTracker,
    get_lr, count_params,
)

# Build model
cfg = ModelConfig()
cfg.vocab_size = VOCAB_SIZE
cfg.chunk_size = CONFIG['window_size']
cfg.max_seq_len = CONFIG['max_seq_len']
# Recalculate text positions for longer sequences
cfg.n_text_positions = cfg.max_seq_len - cfg.n_mem_positions

model = ANT(cfg, use_checkpoint=False).to(device)
n_params = count_params(model)
log_model_summary(model, cfg)

# Compile model for CUDA (significant speedup with inductor backend)
if CONFIG['use_compile']:
    compile_mode = CONFIG.get('compile_mode', 'max-autotune')
    print(f"\n  Compiling model with torch.compile (mode={compile_mode})...")
    model = torch.compile(model, mode=compile_mode)
    # Warmup: run a dummy batch through to trigger compilation
    print("  Warmup compilation...")
    with torch.no_grad(), autocast('cuda', dtype=torch.bfloat16):
        dummy = torch.randint(0, 256, (64, 128), device=device)
        _ = model(dummy)
    torch.cuda.synchronize()
    print("  ✓ Model compiled and warmed up")

In [ ]:
# ═══════════════════════════════════════════════════════════
# Load Data — Wikipedia MARKDOWN + Shell commands
# The agent speaks markdown and shell — train on both.
# ═══════════════════════════════════════════════════════════

print('Loading data...')
t0 = time.time()

# ── QA data ──
print(f"  Generating {CONFIG['n_train']} QA train + {CONFIG['n_val']} val examples...")
train_examples = generate_dataset(
    CONFIG['n_train'], seed=CONFIG['seed'],
    tagged=CONFIG['tagged'], include_conflicts=CONFIG['tagged'])
val_examples = generate_dataset(
    CONFIG['n_val'], seed=CONFIG['seed'] + 1,
    tagged=CONFIG['tagged'], include_conflicts=CONFIG['tagged'])

# ── Shell data ──
print(f"  Generating {CONFIG['n_shell']} shell commands...")
shell_texts = generate_shell_texts(CONFIG['n_shell'], seed=42)

# ── Wikipedia Markdown data ──
# Loads Wikipedia articles and converts wikitext sections to markdown format.
# Headers become ## headers, lists stay as -, paragraphs preserved.

import re

def load_wikipedia_markdown(n=100000, min_len=50, max_len=800, seed=42,
                            cache_dir='data_cache'):
    """Load Wikipedia sections formatted as markdown.
    
    Converts wiki section headers (== X ==) to markdown (## X),
    preserves paragraph structure, filters for quality.
    Returns list of markdown-formatted text chunks.
    """
    cache_path = os.path.join(cache_dir, f'wiki_markdown_{n}.txt')
    if os.path.exists(cache_path):
        print(f'  Loading cached wiki markdown from {cache_path}')
        with open(cache_path, 'r') as f:
            sections = [s.strip() for s in f.read().split('\n---SECTION---\n') if s.strip()]
        if len(sections) >= n:
            return sections[:n]
    
    print(f'  Downloading Wikipedia for {n} markdown sections...')
    os.makedirs(cache_dir, exist_ok=True)
    
    from datasets import load_dataset as hf_load
    random.seed(seed)
    
    try:
        ds = hf_load('wikimedia/wikipedia', '20231101.en', split='train', streaming=True)
        print('  Using wikimedia/wikipedia')
    except Exception as e:
        print(f'  wikimedia failed: {e}, trying fallback...')
        try:
            ds = hf_load('wikipedia', '20220301.en', split='train', streaming=True)
            print('  Using wikipedia/20220301.en')
        except Exception as e2:
            print(f'  All Wikipedia sources failed: {e2}')
            return _generate_fallback_markdown(n, seed)
    
    sections = []
    for article in ds:
        text = article.get('text', '')
        title = article.get('title', '')
        
        # Split into sections by == headers ==
        parts = re.split(r'\n(={2,})\s*([^=]+?)\s*\1\n', text)
        
        # First part is the intro (use article title as header)
        if parts[0].strip() and len(parts[0].strip()) >= min_len:
            intro = f'# {title}\n\n{parts[0].strip()}'[:max_len]
            if len(intro) >= min_len:
                sections.append(intro)
        
        # Subsequent parts come in triples: (=level=, header_text, body)
        i = 1
        while i + 2 <= len(parts):
            level_str = parts[i]     # == or === etc.
            header = parts[i+1].strip()
            body = parts[i+2].strip() if i+2 < len(parts) else ''
            i += 3
            
            if not body or len(body) < min_len:
                continue
            
            # Skip refs/external links/see also sections
            skip_headers = {'references', 'external links', 'see also',
                           'further reading', 'notes', 'bibliography',
                           'sources', 'citations'}
            if header.lower() in skip_headers:
                continue
            
            # Convert == level to markdown # level
            md_level = '#' * len(level_str)
            
            # Clean up body: remove wiki markup remnants
            body = re.sub(r'\[\[(?:[^|\]]*\|)?([^\]]+)\]\]', r'\1', body)  # [[link|text]] → text
            body = re.sub(r'\{\{[^}]+\}\}', '', body)  # remove templates
            body = re.sub(r"'{2,3}", '', body)  # remove bold/italic wiki markup
            body = re.sub(r'<[^>]+>', '', body)  # remove HTML tags
            body = re.sub(r'\n{3,}', '\n\n', body)  # collapse blank lines
            
            # Convert bullet points
            body = re.sub(r'^\*\s+', '- ', body, flags=re.MULTILINE)
            
            section = f'{md_level} {header}\n\n{body}'[:max_len]
            if len(section) >= min_len:
                sections.append(section)
        
        if len(sections) >= n * 2:
            break
    
    random.shuffle(sections)
    sections = sections[:n]
    
    # Cache
    with open(cache_path, 'w') as f:
        f.write('\n---SECTION---\n'.join(sections))
    print(f'  Cached {len(sections)} markdown sections to {cache_path}')
    return sections


def _generate_fallback_markdown(n, seed):
    """Fallback: synthetic markdown if Wikipedia fails."""
    random.seed(seed)
    templates = [
        '## {topic}\n\n{topic} is a {adj} subject in {field}. It involves {desc}.',
        '# {topic}\n\n{intro}\n\n## Overview\n\n{desc}\n\n## Details\n\n- {point1}\n- {point2}',
        '## {topic}\n\n> {quote}\n\n{desc}\n\n### Key Points\n\n1. {point1}\n2. {point2}',
        '## {topic}\n\n{desc}\n\n```\n{code}\n```\n\n{conclusion}',
    ]
    topics = ['Science', 'History', 'Technology', 'Nature', 'Art', 'Mathematics',
              'Geography', 'Physics', 'Chemistry', 'Biology', 'Music', 'Literature']
    adjs = ['fundamental', 'complex', 'fascinating', 'important', 'evolving', 'ancient']
    fields = ['modern research', 'human knowledge', 'academic study', 'daily life']
    texts = []
    for _ in range(n):
        t = random.choice(templates).format(
            topic=random.choice(topics), adj=random.choice(adjs),
            field=random.choice(fields),
            desc='This area has seen significant developments in recent years.',
            intro='An important area of study with wide applications.',
            quote='Knowledge is power.',
            point1='First key observation', point2='Second important finding',
            code='x = compute(data)', conclusion='Further research is needed.')
        texts.append(t)
    return texts


print(f"  Loading {CONFIG['n_wiki_md']} Wikipedia markdown sections...")
wiki_md_texts = load_wikipedia_markdown(
    CONFIG['n_wiki_md'], seed=42,
    min_len=CONFIG['wiki_min_len'],
    max_len=CONFIG['wiki_max_len'])

if CONFIG['tagged']:
    shell_texts = [tag_text(t, domain='shell') for t in shell_texts]
    wiki_md_texts = [tag_text(t, domain='wiki') for t in wiki_md_texts]

all_lm_texts = shell_texts + wiki_md_texts
random.shuffle(all_lm_texts)

# Data statistics
total_bytes = sum(len(t.encode('utf-8')) for t in all_lm_texts)
avg_len = total_bytes / len(all_lm_texts)
elapsed = time.time() - t0

print(f"\n  Data loaded in {elapsed:.0f}s")
print(f"  QA:    {len(train_examples)} train, {len(val_examples)} val")
print(f"  LM:    {len(all_lm_texts)} texts ({total_bytes/1e6:.1f}MB, avg {avg_len:.0f} bytes)")
print(f"  Shell: {len(shell_texts)}, Wiki MD: {len(wiki_md_texts)}")
print(f"\n  Sample wiki markdown:")
print(f"  {wiki_md_texts[0][:200]}...")
print(f"\n  Sample shell:")
print(f"  {shell_texts[0][:200]}")

In [ ]:
# Build DataLoaders — A100 needs HUGE batches to saturate
# With 828K params, the model fits in L2 cache. Throughput scales
# linearly with batch size until we hit memory bandwidth limits.

lm_ds = TextLMDataset(all_lm_texts, max_len=CONFIG['max_seq_len'])

lm_loader = DataLoader(
    lm_ds, batch_size=CONFIG['batch_size'], shuffle=True, drop_last=True,
    num_workers=CONFIG['num_workers'], pin_memory=CONFIG['pin_memory'],
    persistent_workers=True, prefetch_factor=4)

lm_loader_large = DataLoader(
    lm_ds, batch_size=CONFIG['lm_batch_size'], shuffle=True, drop_last=True,
    num_workers=CONFIG['num_workers'], pin_memory=CONFIG['pin_memory'],
    persistent_workers=True, prefetch_factor=4)

# Estimate VRAM usage
param_mem = sum(p.numel() * p.element_size() for p in model.parameters()) / 1e6
batch_mem_est = CONFIG['batch_size'] * CONFIG['max_seq_len'] * 2 / 1e6  # bf16 tokens
print(f"  LM dataset: {len(lm_ds)} samples")
print(f"  LM loader: {len(lm_loader)} batches (micro-B={CONFIG['batch_size']}, eff-B={CONFIG['batch_size'] * CONFIG['grad_accum_steps']})")
print(f"  LM loader (large): {len(lm_loader_large)} batches (micro-B={CONFIG['lm_batch_size']}, eff-B={CONFIG['lm_batch_size'] * CONFIG['lm_grad_accum_steps']})")
print(f"  Model params: {param_mem:.1f} MB")
print(f"  Est. batch tensor: {batch_mem_est:.1f} MB")

# VRAM sanity check
if torch.cuda.is_available():
    total_vram = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"  GPU VRAM: {total_vram:.0f} GB — {'plenty of room' if total_vram > 30 else 'may need smaller batches'}")

## 4. TensorBoard Setup

In [ ]:
from torch.utils.tensorboard import SummaryWriter

log_dir = os.path.join(LOG_DIR, time.strftime('%Y%m%d_%H%M%S'))
writer = SummaryWriter(log_dir=log_dir)
print(f"TensorBoard logs: {log_dir}")

# Launch TensorBoard (Colab magic)
%load_ext tensorboard
%tensorboard --logdir {LOG_DIR}

## 5. Training Utilities

In [ ]:
import glob as glob_mod

def save_checkpoint(model, optimizer, step, metrics, ckpt_dir, tag='latest'):
    """Save checkpoint locally + upload to HF Hub."""
    # Handle compiled model
    m = model._orig_mod if hasattr(model, '_orig_mod') else model
    
    ckpt = {
        'model': m.state_dict(),
        'optimizer': optimizer.state_dict(),
        'step': step,
        'metrics': metrics,
        'config': {
            'd_model': cfg.d_model, 'n_layers': cfg.n_layers,
            'n_heads': cfg.n_heads, 'vocab_size': VOCAB_SIZE,
            'max_seq_len': cfg.max_seq_len,
        },
        'window_size': CONFIG['window_size'],
        'num_passes': CONFIG['num_passes'],
        'mode': 'multitask',
        'vocab': VOCAB,
        'accuracy': metrics.get('qa_acc', 0),
    }
    
    # Save locally
    os.makedirs(ckpt_dir, exist_ok=True)
    path = os.path.join(ckpt_dir, f'checkpoint_{tag}.pt')
    torch.save(ckpt, path)
    
    # Also save numbered checkpoint
    numbered_path = os.path.join(ckpt_dir, f'checkpoint_step{step:06d}.pt')
    torch.save(ckpt, numbered_path)
    
    # Cleanup old numbered checkpoints
    keep_n = CONFIG['keep_last_n_checkpoints']
    numbered = sorted(glob_mod.glob(os.path.join(ckpt_dir, 'checkpoint_step*.pt')))
    for old in numbered[:-keep_n]:
        os.remove(old)
    
    # Upload to HF Hub periodically
    if step % CONFIG.get('hf_upload_interval', 1000) == 0 or tag == 'best':
        try:
            phase_name = os.path.basename(ckpt_dir)
            hf_path = f'checkpoints/{phase_name}/checkpoint_{tag}.pt'
            hf_api.upload_file(
                path_or_fileobj=path,
                path_in_repo=hf_path,
                repo_id=HF_REPO,
                repo_type='model',
            )
            print(f'    ↑ Uploaded to HF: {hf_path}')
        except Exception as e:
            print(f'    ⚠ HF upload failed: {e}')
    
    return path


def load_checkpoint(ckpt_dir, model, optimizer=None, tag='latest'):
    """Load checkpoint, returns step number."""
    path = os.path.join(ckpt_dir, f'checkpoint_{tag}.pt')
    if not os.path.exists(path):
        print(f"  No checkpoint at {path}")
        return 0
    
    ckpt = torch.load(path, map_location=device, weights_only=False)
    m = model._orig_mod if hasattr(model, '_orig_mod') else model
    m.load_state_dict(ckpt['model'])
    if optimizer and 'optimizer' in ckpt:
        optimizer.load_state_dict(ckpt['optimizer'])
    step = ckpt.get('step', 0)
    metrics = ckpt.get('metrics', {})
    print(f"  ✓ Loaded checkpoint step={step}, metrics={metrics}")
    return step


def log_verbose(step, total_steps, phase, losses, tracker, writer, extra=''):
    """Verbose logging to console and TensorBoard."""
    elapsed = time.time() - tracker._last_time if tracker._last_time else 0
    
    # Console
    loss_str = ' '.join(f'{k}={v:.4f}' for k, v in losses.items())
    print(f"  [{phase} {step:>6}/{total_steps}] {loss_str} "
          f"gnorm={tracker.avg_grad_norm:.2f} "
          f"spd={tracker.steps_per_sec:.1f}it/s "
          f"{extra}")
    
    # TensorBoard
    for k, v in losses.items():
        writer.add_scalar(f'{phase}/{k}', v, step)
    writer.add_scalar(f'{phase}/steps_per_sec', tracker.steps_per_sec, step)
    writer.add_scalar(f'{phase}/grad_norm', tracker.avg_grad_norm, step)


@torch.no_grad()
def generate_sample(model, prompt='The ', max_len=100, temperature=0.8):
    """Quick generation sample for monitoring."""
    m = model._orig_mod if hasattr(model, '_orig_mod') else model
    m.eval()
    
    ids = [BOS_ID] + tokenize(prompt)
    input_ids = torch.tensor([ids], dtype=torch.long, device=device)
    
    for _ in range(max_len):
        logits, _ = m(input_ids)
        next_logits = logits[0, -1, :].float() / max(temperature, 0.01)
        
        # Top-k=50
        topk_vals, _ = next_logits.topk(50)
        next_logits[next_logits < topk_vals[-1]] = float('-inf')
        
        probs = F.softmax(next_logits, dim=-1)
        next_id = torch.multinomial(probs, 1).item()
        
        if next_id == EOS_ID:
            break
        ids.append(next_id)
        input_ids = torch.tensor([ids], dtype=torch.long, device=device)
    
    m.train()
    return detokenize(ids[1:])  # skip BOS

print("✓ Training utilities ready")

## 6. Phase A — LM Warmup

In [ ]:
# ═══════════════════════════════════════════════════════════
# Phase A: Warmup LM on QA patterns (passage in context)
# Uses gradient accumulation to saturate A100 with huge batches
# ═══════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("  Phase A: LM Warmup (QA patterns in context)")
print("=" * 60)

steps_a = CONFIG['phase_a_steps']
lr_a = 3e-4
grad_accum = CONFIG['grad_accum_steps']
micro_bs = CONFIG['batch_size']

print(f"  Steps: {steps_a}, micro-batch: {micro_bs}, accum: {grad_accum}")
print(f"  Effective batch: {micro_bs * grad_accum}")
print(f"  Tokens/step: {micro_bs * grad_accum * CONFIG['max_seq_len']:,}")

qa_ds = ContextQADataset(train_examples, max_len=CONFIG['max_seq_len'])
qa_loader = DataLoader(
    qa_ds, batch_size=micro_bs, shuffle=True, drop_last=True,
    num_workers=CONFIG['num_workers'], pin_memory=CONFIG['pin_memory'],
    persistent_workers=True, prefetch_factor=4)
qa_iter = iter(qa_loader)

optimizer = torch.optim.AdamW(model.parameters(), lr=lr_a, weight_decay=0.1)
scaler = GradScaler('cuda') if CONFIG['use_bf16'] else None

model.train()
tracker = TrainingTracker(window=50)
t0 = time.time()

ckpt_dir_a = os.path.join(LOCAL_DIR, 'phase_a')
os.makedirs(ckpt_dir_a, exist_ok=True)

for step in range(1, steps_a + 1):
    tracker.tick()
    
    lr_now = get_lr(step, 100, steps_a, lr_a, lr_a * 0.01)
    for pg in optimizer.param_groups:
        pg['lr'] = lr_now
    
    optimizer.zero_grad(set_to_none=True)
    accum_loss = 0.0
    
    for accum_i in range(grad_accum):
        try:
            inp, tgt = next(qa_iter)
        except StopIteration:
            qa_iter = iter(qa_loader)
            inp, tgt = next(qa_iter)
        
        inp = inp.to(device, non_blocking=True)
        tgt = tgt.to(device, non_blocking=True)
        
        if CONFIG['use_bf16']:
            with autocast('cuda', dtype=torch.bfloat16):
                logits, _ = model(inp)
                loss = F.cross_entropy(
                    logits.reshape(-1, VOCAB_SIZE), tgt.reshape(-1),
                    ignore_index=PAD_ID) / grad_accum
            scaler.scale(loss).backward()
        else:
            logits, _ = model(inp)
            loss = F.cross_entropy(
                logits.reshape(-1, VOCAB_SIZE), tgt.reshape(-1),
                ignore_index=PAD_ID) / grad_accum
            loss.backward()
        accum_loss += loss.item()
    
    if CONFIG['use_bf16']:
        scaler.unscale_(optimizer)
        grad_norm, _ = log_gradient_stats(model._orig_mod if hasattr(model, '_orig_mod') else model)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
    else:
        grad_norm, _ = log_gradient_stats(model._orig_mod if hasattr(model, '_orig_mod') else model)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
    
    tracker.add_loss(accum_loss)
    tracker.add_grad_norm(grad_norm)
    
    if step % CONFIG['log_interval'] == 0 or step == 1:
        elapsed = time.time() - t0
        eta = elapsed / step * (steps_a - step)
        tok_per_sec = (step * micro_bs * grad_accum * CONFIG['max_seq_len']) / elapsed
        vram_gb = torch.cuda.max_memory_allocated() / 1e9 if torch.cuda.is_available() else 0
        log_verbose(step, steps_a, 'A', {'loss': accum_loss}, tracker, writer,
                   f'lr={lr_now:.1e} {tok_per_sec/1e3:.0f}K tok/s VRAM={vram_gb:.1f}GB [{elapsed:.0f}s / ETA {eta:.0f}s]')
    
    if step % CONFIG['checkpoint_interval'] == 0:
        save_checkpoint(model, optimizer, step,
                       {'loss': tracker.avg_loss}, ckpt_dir_a)

elapsed = time.time() - t0
tok_total = steps_a * micro_bs * grad_accum * CONFIG['max_seq_len']
print(f"\n  Phase A done in {elapsed:.0f}s")
print(f"  {tok_total/1e6:.1f}M tokens, {tok_total/elapsed/1e3:.0f}K tok/s")
print(f"  Final loss={accum_loss:.4f}")
if torch.cuda.is_available():
    print(f"  Peak VRAM: {torch.cuda.max_memory_allocated()/1e9:.1f} GB")

# Quick eval
m = model._orig_mod if hasattr(model, '_orig_mod') else model
ctx_acc = evaluate_context_qa(m, cfg, val_examples, device)
print(f"  Context QA accuracy: {ctx_acc:.1%}")
writer.add_scalar('eval/context_qa', ctx_acc, step)

save_checkpoint(model, optimizer, steps_a,
               {'loss': tracker.avg_loss, 'context_qa': ctx_acc},
               ckpt_dir_a, tag='final')
print("  ✓ Phase A checkpoint saved")

## 7. Multi-Task Training (QA + LM)

In [ ]:
# ═══════════════════════════════════════════════════════════
# Multi-Task Phase: QA + LM simultaneously
# ═══════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("  Multi-Task Training: LM + Memory QA")
print("=" * 60)

steps_mt = CONFIG['multitask_steps']
window_size = CONFIG['window_size']
num_passes = CONFIG['num_passes']
lm_weight = CONFIG['lm_weight']
qa_weight = CONFIG['qa_weight']
qa_micro_bs = CONFIG['qa_batch_size']
lm_micro_bs = CONFIG['batch_size']
grad_accum = CONFIG['grad_accum_steps']

print(f"  Steps:       {steps_mt}")
print(f"  Window:      W={window_size}, P={num_passes}")
print(f"  Weights:     LM={lm_weight}, QA={qa_weight}")
print(f"  LM micro-B:  {lm_micro_bs} × {grad_accum} accum = {lm_micro_bs * grad_accum}")
print(f"  QA micro-B:  {qa_micro_bs}")
print(f"  BF16:        {CONFIG['use_bf16']}")

# Pre-tokenize QA data
pad = PAD_ID
bos = BOS_ID
eos = EOS_ID
ans_marker = ANS_ID
max_passage_len = 256 if CONFIG['tagged'] else 128

qa_data = []
for ex in train_examples:
    passage_ids = tokenize(ex.passage)
    question_ids = tokenize(ex.question)
    answer_ids = tokenize(ex.answer)
    full_seq = [bos, ans_marker] + question_ids + answer_ids + [eos]
    inp_ids = full_seq[:-1]
    n_context = 2 + len(question_ids)
    tgt_ids = [pad] * (n_context - 1) + answer_ids + [eos]
    assert len(inp_ids) == len(tgt_ids)
    p_ids = passage_ids[:max_passage_len]
    while len(p_ids) < max_passage_len:
        p_ids.append(pad)
    qa_data.append((inp_ids, tgt_ids, p_ids))

print(f"  QA data: {len(qa_data)} examples pre-tokenized")

# Optimizer (fresh for this phase)
optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG['lr'], weight_decay=0.1)
scaler = GradScaler('cuda') if CONFIG['use_bf16'] else None
lm_iter = iter(lm_loader)

model.train()
tracker = TrainingTracker(window=50)
t0 = time.time()

best_acc = 0.0
best_step = 0
d1_cutoff = int(steps_mt * 0.3)  # 30% frozen encoder (proven ratio)

ckpt_dir_mt = os.path.join(LOCAL_DIR, 'multitask')
os.makedirs(ckpt_dir_mt, exist_ok=True)

print(f"\n  D1 (frozen encoder): steps 1-{d1_cutoff}")
print(f"  D2 (co-adaptation): steps {d1_cutoff+1}-{steps_mt}")
print(f"  {'─' * 56}")

In [ ]:
# ═══════════════════════════════════════════════════════════
# Multi-Task Training Loop — A100 optimized
# LM: gradient accumulation over large micro-batches
# QA: large batched sliding window encoding
# ═══════════════════════════════════════════════════════════

# Unwrap compiled model for encoding functions
m_raw = model._orig_mod if hasattr(model, '_orig_mod') else model

for step in range(1, steps_mt + 1):
    tracker.tick()
    
    lr_now = get_lr(step, 200, steps_mt, CONFIG['lr'], CONFIG['lr'] * 0.01)
    for pg in optimizer.param_groups:
        pg['lr'] = lr_now
    
    optimizer.zero_grad(set_to_none=True)
    
    # ── LM loss with gradient accumulation ──
    lm_loss_accum = 0.0
    for accum_i in range(grad_accum):
        try:
            lm_inp, lm_tgt = next(lm_iter)
        except StopIteration:
            lm_iter = iter(lm_loader)
            lm_inp, lm_tgt = next(lm_iter)
        lm_inp = lm_inp.to(device, non_blocking=True)
        lm_tgt = lm_tgt.to(device, non_blocking=True)
        
        if CONFIG['use_bf16']:
            with autocast('cuda', dtype=torch.bfloat16):
                lm_logits, _ = model(lm_inp)
                lm_loss = F.cross_entropy(
                    lm_logits.reshape(-1, VOCAB_SIZE), lm_tgt.reshape(-1),
                    ignore_index=pad)
                scaled_lm = (lm_weight * lm_loss) / grad_accum
            scaler.scale(scaled_lm).backward()
        else:
            lm_logits, _ = model(lm_inp)
            lm_loss = F.cross_entropy(
                lm_logits.reshape(-1, VOCAB_SIZE), lm_tgt.reshape(-1),
                ignore_index=pad)
            scaled_lm = (lm_weight * lm_loss) / grad_accum
            scaled_lm.backward()
        lm_loss_accum += lm_loss.item() / grad_accum
    
    # ── QA loss (single large batch, sliding window) ──
    batch_idx = random.sample(range(len(qa_data)), qa_micro_bs)
    batch = [qa_data[i] for i in batch_idx]
    
    max_seq_len = max(len(b[0]) for b in batch)
    inp_batch, tgt_batch, passage_batch = [], [], []
    for inp_ids, tgt_ids, p_ids in batch:
        inp_batch.append(inp_ids + [pad] * (max_seq_len - len(inp_ids)))
        tgt_batch.append(tgt_ids + [pad] * (max_seq_len - len(tgt_ids)))
        passage_batch.append(p_ids)
    
    q_tensor = torch.tensor(inp_batch, dtype=torch.long, device=device)
    tgt_tensor = torch.tensor(tgt_batch, dtype=torch.long, device=device)
    p_tensor = torch.tensor(passage_batch, dtype=torch.long, device=device)
    
    if CONFIG['use_bf16']:
        # Memory encoding in fp32 — addresses need full precision
        with autocast('cuda', enabled=False):
            if step >= d1_cutoff:
                mem_keys, mem_vals, mem_mask = encode_sentence_differentiable(
                    m_raw, p_tensor, device)
            else:
                m_raw.eval()
                mem_keys, mem_vals, mem_mask = encode_sentence_frozen(
                    m_raw, p_tensor, device)
                m_raw.train()
        
        with autocast('cuda', dtype=torch.bfloat16):
            qa_hidden = sliding_lm_encode(
                m_raw, q_tensor, window_size, num_passes,
                mem_keys=mem_keys, mem_vals=mem_vals, mem_mask=mem_mask)
            qa_logits = F.linear(qa_hidden, m_raw.embed.weight)
            qa_loss = F.cross_entropy(
                qa_logits.reshape(-1, VOCAB_SIZE), tgt_tensor.reshape(-1),
                ignore_index=pad)
            scaled_qa = qa_weight * qa_loss
        scaler.scale(scaled_qa).backward()
        
        scaler.unscale_(optimizer)
        grad_norm, _ = log_gradient_stats(m_raw)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
    else:
        if step >= d1_cutoff:
            mem_keys, mem_vals, mem_mask = encode_sentence_differentiable(
                m_raw, p_tensor, device)
        else:
            m_raw.eval()
            mem_keys, mem_vals, mem_mask = encode_sentence_frozen(
                m_raw, p_tensor, device)
            m_raw.train()
        
        qa_hidden = sliding_lm_encode(
            m_raw, q_tensor, window_size, num_passes,
            mem_keys=mem_keys, mem_vals=mem_vals, mem_mask=mem_mask)
        qa_logits = F.linear(qa_hidden, m_raw.embed.weight)
        qa_loss = F.cross_entropy(
            qa_logits.reshape(-1, VOCAB_SIZE), tgt_tensor.reshape(-1),
            ignore_index=pad)
        scaled_qa = qa_weight * qa_loss
        scaled_qa.backward()
        
        grad_norm, _ = log_gradient_stats(m_raw)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
    
    total_loss_val = lm_loss_accum * lm_weight + qa_loss.item() * qa_weight
    tracker.add_loss(total_loss_val)
    tracker.add_grad_norm(grad_norm)
    
    # D2 transition — reset patience so early stopping gives D2 a fair chance
    if step == d1_cutoff:
        print(f'\n  ═══ D1→D2 transition at step {step} ═══')
        print(f'  Encoder now differentiable — gradients flow through memory')
        best_step = step  # Reset patience counter for D2
    
    # Logging
    if step % CONFIG['log_interval'] == 0 or step == 1:
        elapsed = time.time() - t0
        eta = elapsed / step * (steps_mt - step)
        phase_tag = 'D1' if step < d1_cutoff else 'D2'
        lm_tok = step * lm_micro_bs * grad_accum * CONFIG['max_seq_len']
        vram_gb = torch.cuda.max_memory_allocated() / 1e9 if torch.cuda.is_available() else 0
        log_verbose(step, steps_mt, phase_tag,
                   {'total': total_loss_val, 'lm': lm_loss_accum, 'qa': qa_loss.item()},
                   tracker, writer,
                   f'lr={lr_now:.1e} {lm_tok/elapsed/1e3:.0f}K tok/s VRAM={vram_gb:.1f}GB [{elapsed:.0f}s / ETA {eta:.0f}s]')
    
    # Evaluation
    if step % CONFIG['eval_interval'] == 0 or step == steps_mt:
        # Memory quality diagnostic (first eval only)
        if step == CONFIG['eval_interval']:
            diag_idx = random.sample(range(len(qa_data)), min(8, len(qa_data)))
            diag_batch = [qa_data[i] for i in diag_idx]
            diag_p = torch.tensor([b[2] for b in diag_batch], dtype=torch.long, device=device)
            with torch.no_grad():
                dk, dv, dm = encode_sentence_frozen(m_raw, diag_p, device)
            print(f'  ╔══ MEMORY DIAGNOSTIC ══')
            print(f'  ║ Keys:  shape={tuple(dk.shape)}, mean={dk.mean():.4f}, std={dk.std():.4f}, zeros={int((dk.abs() < 1e-6).all(-1).sum())}')
            print(f'  ║ Vals:  mean={dv.mean():.4f}, std={dv.std():.4f}')
            print(f'  ║ Mask:  {dm.sum().item()}/{dm.numel()} valid ({dm.float().mean():.1%})')
            print(f'  ║ dtype: keys={dk.dtype}, vals={dv.dtype}')
            print(f'  ╚{"═" * 40}')
        
        acc, breakdown = evaluate_sliding_lm(
            m_raw, cfg, val_examples, device,
            window_size=window_size, num_passes=num_passes,
            encode_fn=encode_sentence_frozen)
        tracker.add_eval(step, acc, breakdown)
        
        delta = acc - tracker.eval_history[-2][1] if len(tracker.eval_history) > 1 else 0
        delta_str = f" ({'+' if delta >= 0 else ''}{delta:.1%})" if len(tracker.eval_history) > 1 else ''
        
        print(f"\n  ╔══ EVAL @ step {step} ══")
        print(f"  ║ QA Accuracy: {acc:.1%}{delta_str}  (best: {tracker.best_acc:.1%})")
        print(f"  ║ LM loss: {lm_loss_accum:.4f}  QA loss: {qa_loss.item():.4f}")
        if breakdown:
            for n_facts, (corr, tot) in sorted(breakdown.items()):
                bar = '█' * int(corr / max(tot, 1) * 20)
                print(f"  ║   {n_facts}-fact: {corr:>3}/{tot:>3} = {corr/max(tot,1):.1%} {bar}")
        
        sample = generate_sample(model, '## ', max_len=80, temperature=0.8)
        print(f"  ║ Sample: {sample[:120]}")
        print(f"  ╚{'═' * 40}\n")
        
        writer.add_scalar('eval/qa_accuracy', acc, step)
        writer.add_text('eval/sample', sample[:200], step)
        
        if acc > best_acc:
            best_acc = acc
            best_step = step
            save_checkpoint(model, optimizer, step,
                          {'qa_acc': acc, 'lm_loss': lm_loss_accum},
                          ckpt_dir_mt, tag='best')
            print(f"  ✓ New best! QA={acc:.1%} saved")
        
        # Early stopping (only in D2, with grace period)
        if step > d1_cutoff + CONFIG['eval_interval'] * 2 and step - best_step >= CONFIG['eval_interval'] * 5:
            print(f"\n  ⚠ Early stopping at step {step} (no improvement since {best_step})")
            load_checkpoint(ckpt_dir_mt, model, tag='best')
            break
    
    # Periodic checkpoint
    if step % CONFIG['checkpoint_interval'] == 0:
        save_checkpoint(model, optimizer, step,
                       {'qa_acc': best_acc, 'lm_loss': lm_loss_accum},
                       ckpt_dir_mt)

elapsed = time.time() - t0
print(f"\n  Multi-task done in {elapsed:.0f}s ({elapsed/60:.1f} min)")
print(f"  Best QA: {best_acc:.1%} at step {best_step}")
if torch.cuda.is_available():
    print(f"  Peak VRAM: {torch.cuda.max_memory_allocated()/1e9:.1f} GB")

## 8. Extended LM Training (Fluency Phase)

In [ ]:
# ═══════════════════════════════════════════════════════════
# Phase LM: Extended pure LM training for fluency
# QA has converged — now focus purely on language modeling
# with much more data and larger batches
# ═══════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("  Extended LM Training — Fluency Phase")
print("=" * 60)

steps_lm = CONFIG['lm_only_steps']
lm_lr = CONFIG['lm_lr']
lm_batch = CONFIG['lm_batch_size']
lm_accum = CONFIG['lm_grad_accum_steps']

print(f"  Steps:    {steps_lm}")
print(f"  Batch:    {lm_batch} × {lm_accum} accum = {lm_batch * lm_accum}")
print(f"  Tok/step: {lm_batch * lm_accum * CONFIG['max_seq_len']:,}")
print(f"  LR:       {lm_lr}")
print(f"  LM data:  {len(all_lm_texts)} texts")

# Fresh optimizer for LM phase
optimizer_lm = torch.optim.AdamW(model.parameters(), lr=lm_lr, weight_decay=0.05)
scaler = GradScaler('cuda') if CONFIG['use_bf16'] else None
lm_iter_large = iter(lm_loader_large)

model.train()
tracker_lm = TrainingTracker(window=50)
t0 = time.time()
best_lm_loss = float('inf')

ckpt_dir_lm = os.path.join(LOCAL_DIR, 'lm_fluency')
os.makedirs(ckpt_dir_lm, exist_ok=True)

for step in range(1, steps_lm + 1):
    tracker_lm.tick()
    
    lr_now = get_lr(step, 500, steps_lm, lm_lr, lm_lr * 0.01)
    for pg in optimizer_lm.param_groups:
        pg['lr'] = lr_now
    
    optimizer_lm.zero_grad(set_to_none=True)
    accum_loss = 0.0
    
    for accum_i in range(lm_accum):
        try:
            lm_inp, lm_tgt = next(lm_iter_large)
        except StopIteration:
            lm_iter_large = iter(lm_loader_large)
            lm_inp, lm_tgt = next(lm_iter_large)
        
        lm_inp = lm_inp.to(device, non_blocking=True)
        lm_tgt = lm_tgt.to(device, non_blocking=True)
        
        if CONFIG['use_bf16']:
            with autocast('cuda', dtype=torch.bfloat16):
                logits, _ = model(lm_inp)
                loss = F.cross_entropy(
                    logits.reshape(-1, VOCAB_SIZE), lm_tgt.reshape(-1),
                    ignore_index=PAD_ID) / lm_accum
            scaler.scale(loss).backward()
        else:
            logits, _ = model(lm_inp)
            loss = F.cross_entropy(
                logits.reshape(-1, VOCAB_SIZE), lm_tgt.reshape(-1),
                ignore_index=PAD_ID) / lm_accum
            loss.backward()
        accum_loss += loss.item()
    
    if CONFIG['use_bf16']:
        scaler.unscale_(optimizer_lm)
        grad_norm, _ = log_gradient_stats(
            model._orig_mod if hasattr(model, '_orig_mod') else model)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer_lm)
        scaler.update()
    else:
        grad_norm, _ = log_gradient_stats(
            model._orig_mod if hasattr(model, '_orig_mod') else model)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer_lm.step()
    
    tracker_lm.add_loss(accum_loss)
    tracker_lm.add_grad_norm(grad_norm)
    bpb = accum_loss / math.log(2)
    
    if step % CONFIG['log_interval'] == 0 or step == 1:
        elapsed = time.time() - t0
        eta = elapsed / step * (steps_lm - step)
        tok_per_sec = (step * lm_batch * lm_accum * CONFIG['max_seq_len']) / elapsed
        vram_gb = torch.cuda.max_memory_allocated() / 1e9 if torch.cuda.is_available() else 0
        log_verbose(step, steps_lm, 'LM',
                   {'loss': accum_loss, 'bpb': bpb},
                   tracker_lm, writer,
                   f'lr={lr_now:.1e} {tok_per_sec/1e3:.0f}K tok/s VRAM={vram_gb:.1f}GB [{elapsed:.0f}s / ETA {eta:.0f}s]')
    
    # Sample generation
    if step % (CONFIG['eval_interval'] * 2) == 0 or step == 1:
        sample = generate_sample(model, '## ', max_len=100, temperature=0.8)
        print(f"  ┌── Sample @ step {step}:")
        print(f"  │ {sample[:200]}")
        print(f"  └── bits/byte={bpb:.3f}")
        writer.add_text('lm/sample', sample[:200], step + steps_mt)
        
        # Also check QA hasn't degraded
        m_raw = model._orig_mod if hasattr(model, '_orig_mod') else model
        acc, _ = evaluate_sliding_lm(
            m_raw, cfg, val_examples[:200], device,
            window_size=window_size, num_passes=num_passes,
            encode_fn=encode_sentence_frozen)
        print(f"  QA check: {acc:.1%}")
        writer.add_scalar('eval/qa_during_lm', acc, step + steps_mt)
    
    # Checkpoint
    if step % CONFIG['checkpoint_interval'] == 0:
        if tracker_lm.avg_loss < best_lm_loss:
            best_lm_loss = tracker_lm.avg_loss
            save_checkpoint(model, optimizer_lm, step + steps_mt,
                          {'lm_loss': tracker_lm.avg_loss, 'bpb': bpb},
                          ckpt_dir_lm, tag='best')
        save_checkpoint(model, optimizer_lm, step + steps_mt,
                       {'lm_loss': tracker_lm.avg_loss, 'bpb': bpb},
                       ckpt_dir_lm)

elapsed = time.time() - t0
tok_total = steps_lm * lm_batch * lm_accum * CONFIG['max_seq_len']
print(f"\n  LM phase done in {elapsed:.0f}s ({elapsed/60:.1f} min)")
print(f"  {tok_total/1e9:.2f}B tokens processed, {tok_total/elapsed/1e3:.0f}K tok/s")
print(f"  Final bits/byte: {bpb:.3f}")
print(f"  Best avg LM loss: {best_lm_loss:.4f}")
if torch.cuda.is_available():
    print(f"  Peak VRAM: {torch.cuda.max_memory_allocated()/1e9:.1f} GB")

## 9. Final Evaluation & Export

In [ ]:
# ═══════════════════════════════════════════════════════════
# Final comprehensive evaluation
# ═══════════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("  Final Evaluation")
print("=" * 60)

m_raw = model._orig_mod if hasattr(model, '_orig_mod') else model
m_raw.eval()

# QA accuracy on full val set
print("\n  Memory QA (full validation set):")
final_acc, final_breakdown = evaluate_sliding_lm(
    m_raw, cfg, val_examples, device,
    max_examples=len(val_examples),
    window_size=window_size, num_passes=num_passes,
    encode_fn=encode_sentence_frozen)
print(f"  → {final_acc:.1%}")

# Fresh test set
print("\n  Memory QA (fresh test set, seed=777):")
test_examples = generate_dataset(
    500, seed=777,
    tagged=CONFIG['tagged'], include_conflicts=CONFIG['tagged'])
test_acc, test_breakdown = evaluate_sliding_lm(
    m_raw, cfg, test_examples, device,
    max_examples=500,
    window_size=window_size, num_passes=num_passes,
    encode_fn=encode_sentence_frozen)
print(f"  → {test_acc:.1%}")

# LM quality
print("\n  LM Quality:")
lm_test_loader = DataLoader(
    lm_ds, batch_size=64, shuffle=False, drop_last=True)
total_lm_loss = 0
n_batches = 0
with torch.no_grad():
    for inp, tgt in lm_test_loader:
        inp, tgt = inp.to(device), tgt.to(device)
        logits, _ = m_raw(inp)
        loss = F.cross_entropy(
            logits.reshape(-1, VOCAB_SIZE), tgt.reshape(-1),
            ignore_index=PAD_ID)
        total_lm_loss += loss.item()
        n_batches += 1
        if n_batches >= 50:
            break

avg_lm_loss = total_lm_loss / n_batches
bpb = avg_lm_loss / math.log(2)
print(f"  Cross-entropy: {avg_lm_loss:.4f}")
print(f"  Bits/byte: {bpb:.3f}")
print(f"  Perplexity: {math.exp(avg_lm_loss):.1f}")

# Generation samples
print("\n  Generation Samples:")
for prompt in ['## ', '# Overview\n\n', '```\n', '- ', 'The ']:
    sample = generate_sample(model, prompt, max_len=80, temperature=0.8)
    print(f"    '{prompt}' → {sample[:150]}")

# Save final report
report = {
    'val_qa_accuracy': final_acc,
    'test_qa_accuracy': test_acc,
    'lm_cross_entropy': avg_lm_loss,
    'lm_bits_per_byte': bpb,
    'lm_perplexity': math.exp(avg_lm_loss),
    'n_params': n_params,
    'config': CONFIG,
    'breakdown': {str(k): v for k, v in final_breakdown.items()},
}
report_path = os.path.join(LOCAL_DIR, 'final_report.json')
with open(report_path, 'w') as f:
    json.dump(report, f, indent=2)
print(f"\n  Report saved to {report_path}")

# Upload report to HF Hub
try:
    hf_api.upload_file(
        path_or_fileobj=report_path,
        path_in_repo='final_report.json',
        repo_id=HF_REPO, repo_type='model')
    print(f'  ↑ Report uploaded to HF: {HF_REPO}')
except Exception as e:
    print(f'  ⚠ HF upload: {e}')

In [ ]:
# ═══════════════════════════════════════════════════════════
# Export best checkpoint for local use
# ═══════════════════════════════════════════════════════════

# Save a clean inference-only checkpoint (smaller, no optimizer state)
m_raw = model._orig_mod if hasattr(model, '_orig_mod') else model

export_ckpt = {
    'model': m_raw.state_dict(),
    'step': 'final',
    'accuracy': test_acc,
    'vocab': VOCAB,
    'window_size': CONFIG['window_size'],
    'num_passes': CONFIG['num_passes'],
    'mode': 'multitask',
    'config': {
        'd_model': cfg.d_model, 'n_layers': cfg.n_layers,
        'n_heads': cfg.n_heads, 'vocab_size': VOCAB_SIZE,
        'max_seq_len': cfg.max_seq_len,
    },
    'lm_bits_per_byte': bpb,
}

export_path = os.path.join(LOCAL_DIR, 'best_multitask.pt')
torch.save(export_ckpt, export_path)
size_mb = os.path.getsize(export_path) / 1e6
print(f"  Exported inference checkpoint: {export_path} ({size_mb:.1f} MB)")
print(f"  QA accuracy: {test_acc:.1%}")
print(f"  LM bits/byte: {bpb:.3f}")

# Push final model to HF Hub
try:
    hf_api.upload_file(
        path_or_fileobj=export_path,
        path_in_repo='best_multitask.pt',
        repo_id=HF_REPO, repo_type='model')
    print(f'\n  ✓ Final model uploaded to: https://huggingface.co/{HF_REPO}')
    print(f'  Download: huggingface-cli download {HF_REPO} best_multitask.pt')
except Exception as e:
    print(f'  ⚠ HF upload: {e}')

print(f"\n  Local use: python chat.py")

## 10. Interactive Chat Test (in Colab)

In [ ]:
# Quick interactive test in notebook
from IPython.display import display, HTML

m_raw = model._orig_mod if hasattr(model, '_orig_mod') else model
m_raw.eval()

def chat_generate(prompt, max_tokens=150, temperature=0.8, top_k=50):
    """Generate text from a prompt."""
    ids = [BOS_ID] + tokenize(prompt)
    input_ids = torch.tensor([ids], dtype=torch.long, device=device)
    
    for _ in range(max_tokens):
        with torch.no_grad():
            logits, _ = m_raw(input_ids)
        next_logits = logits[0, -1, :].float() / max(temperature, 0.01)
        
        # Top-k
        if top_k > 0:
            topk_vals, _ = next_logits.topk(top_k)
            next_logits[next_logits < topk_vals[-1]] = float('-inf')
        
        # Repetition penalty
        for prev_id in set(ids[-30:]):
            if next_logits[prev_id] > 0:
                next_logits[prev_id] /= 1.2
            else:
                next_logits[prev_id] *= 1.2
        
        probs = F.softmax(next_logits, dim=-1)
        next_id = torch.multinomial(probs, 1).item()
        
        if next_id == EOS_ID:
            break
        ids.append(next_id)
        input_ids = torch.tensor([ids], dtype=torch.long, device=device)
    
    return detokenize(ids[1:])


def chat_qa(passage, question):
    """Memory-backed QA."""
    p_ids = tokenize(passage)
    p_tensor = torch.tensor([p_ids], dtype=torch.long, device=device)
    with torch.no_grad():
        mem_keys, mem_vals, mem_mask = encode_sentence_frozen(
            m_raw, p_tensor, device)
    
    q_ids = tokenize(question)
    prompt = [BOS_ID, ANS_ID] + q_ids
    input_ids = torch.tensor([prompt], dtype=torch.long, device=device)
    
    generated = list(prompt)
    for _ in range(50):
        inp = torch.tensor([generated], dtype=torch.long, device=device)
        hidden = sliding_lm_encode(
            m_raw, inp, CONFIG['window_size'], CONFIG['num_passes'],
            mem_keys=mem_keys, mem_vals=mem_vals, mem_mask=mem_mask)
        logits = F.linear(hidden, m_raw.embed.weight)
        next_id = logits[0, -1, :].argmax().item()
        if next_id == EOS_ID or next_id == PAD_ID:
            break
        generated.append(next_id)
    
    return detokenize(generated[len(prompt):])


# Test it!
print("═" * 60)
print("LM Generation:")
for p in ['## ', '# History\n\nThe ', '```bash\n']:
    result = chat_generate(p)
    print(f"  {p} → {result[:200]}")

print("\nMemory QA:")
passage = "mary went to the garden . john went to the kitchen ."
for q in ["where is mary ?", "where is john ?"]:
    answer = chat_qa(passage, q)
    print(f"  Passage: {passage}")
    print(f"  Q: {q} → A: {answer}")

In [ ]:
# Cleanup
writer.close()
print("\n✓ Training complete!")
print(f"  Checkpoints: {LOCAL_DIR}/")
print(f"  HF Hub: https://huggingface.co/{HF_REPO}")
print(f"  Download: huggingface-cli download {HF_REPO} best_multitask.pt")
print(f"  Then run: python chat.py")